# Agente de inscripciones — explicación en clase

Esta notebook reproduce `agente-referencia/agent.py` por bloques. Cada celda de código contiene el mismo código que el archivo, salvo el bloque `if __name__ == "__main__":`, que no se ejecuta en Jupyter.

La idea es recorrer el agente de afuera hacia adentro:

1. Arquitectura y flujo de mensajes.
2. Configuración y system prompt.
3. Herramientas y capa HTTP.
4. Dispatch, cliente de Ollama y bucle.
5. Entrada, verificación de servicios y demo con la API y Ollama reales.

## Preparación de la demo

La notebook usa servicios reales en `localhost:8008` y `localhost:11434`. Antes de ejecutarla:

```bash
cd ../sistema-inscripciones
docker compose up --build -d
docker compose exec api python -m app.seed --reset

cd ../ollama
docker compose up -d --wait
docker compose exec ollama ollama pull qwen3.5:9b
```

El kernel de Python necesita el paquete `requests`.

La demo usa el legajo `1001`. Con el seed determinista del TP, ese alumno es Ana Pérez y tiene aprobadas `CNC`, `CNM`, `SI101`, `SI102`, `SI208`, `SI308` y `SI409`.

## 1. Arquitectura

```mermaid
flowchart LR
    C["Cliente (VSCodium)"] -->|mensaje| M["main"]
    M -->|messages| R["responder"]
    R -->|POST /api/chat| O["Ollama"]
    O -->|assistant + tool_calls| R
    R -->|tool_call| E["ejecutar_herramienta"]
    E -->|HTTP| A["API de inscripciones"]
    A -->|JSON| E
    E -->|resultado| R
    R -->|texto final| M
    M -->|respuesta| C
```

| Componente | Responsabilidad |
|---|---|
| `main` | Lee la entrada, mantiene el historial y muestra la respuesta |
| `responder` | Coordina las llamadas al modelo y la ejecución de herramientas |
| `_ollama_chat` | Habla con Ollama por HTTP |
| `ejecutar_herramienta` | Traduce un `tool_call` a una operación concreta |
| `_api` | Centraliza las llamadas HTTP a la API |
| `_system_prompt` | Arma el mensaje de sistema con el legajo actual |
| `_verificar_servicios` | Comprueba que la API y Ollama estén disponibles |

La idea central: **el modelo decide, el código ejecuta**. El modelo no llama a la API directamente; solo produce un `tool_call` con nombre y argumentos. El código Python ejecuta la herramienta, obtiene el resultado y se lo devuelve al modelo como un mensaje más.

### Secuencia de un turno

```mermaid
sequenceDiagram
    autonumber
    participant C as Cliente
    participant M as main
    participant R as responder
    participant O as Ollama
    participant E as ejecutar_herramienta
    participant A as API

    C->>M: escribe un mensaje
    M->>M: messages.append(user)
    M->>R: responder(messages)
    R->>O: POST /api/chat (historial + tools)
    O-->>R: assistant con tool_calls
    R->>R: messages.append(assistant)
    R->>E: nombre + argumentos
    E->>A: petición HTTP
    A-->>E: JSON
    E-->>R: resultado
    R->>R: messages.append(tool)
    R->>O: POST /api/chat (historial + resultado)
    O-->>R: assistant final
    R-->>M: texto
    M-->>C: imprime la respuesta
```

El turno completo se entiende como una conversación entre seis participantes. `main` solo lee y escribe; `responder` decide cuándo volver a llamar al modelo; `ejecutar_herramienta` y `_api` son la parte que toca el mundo exterior.

In [1]:
import json
import os
import sys
import unicodedata
from urllib.parse import quote

import requests

## 2. Configuración (`agent.py:9-14`)

| Constante | Valor por defecto | Efecto |
|---|---|---|
| `API_URL` | `http://localhost:8008` | Base de la API de inscripciones |
| `OLLAMA_URL` | `http://localhost:11434` | Base de Ollama |
| `MODEL` | `qwen3.5:9b` | Modelo que hace el tool calling |
| `LEGAJO_ACTUAL` | `None` | Legajo que el prompt usa por defecto |
| `MAX_PASOS` | `8` | Cantidad máxima de llamadas al modelo por turno |

`LEGAJO_ACTUAL` no reemplaza al legajo que el usuario escribe: solo agrega una nota al mensaje de sistema. `MAX_PASOS` cuenta llamadas al modelo, no herramientas: una vuelta puede ejecutar varias `tool_calls`.

In [3]:
# Configuración: editar acá.
API_URL = "http://localhost:8008"
OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3.5:9b"
LEGAJO_ACTUAL = None
MAX_PASOS = 8

## 3. System prompt (`agent.py:16-30`)

El mensaje de sistema define el rol y las restricciones del asistente:

- responde en español, breve y claro;
- usa las herramientas para consultar datos reales y no inventa datos;
- para inscribir necesita legajo, materia y turno;
- si la API rechaza una inscripción, la respuesta final incluye el código exacto del error;
- para dar de baja, primero lista las inscripciones y usa el `id` exacto;
- no crea, edita ni borra alumnos o materias, ni resetea la base.

Los códigos de error que menciona el prompt son:

```text
MATERIA_APROBADA
SIN_CUPO
CORRELATIVA_PENDIENTE
CHOQUE_TURNO
INSCRIPCION_DUPLICADA
TURNO_NO_DISPONIBLE
```

El texto `{legajo}` se completa después en `_system_prompt()`. El prompt viaja como primer mensaje de la conversación y el modelo lo ve en cada llamada.

In [4]:
SYSTEM_PROMPT = """Sos un asistente de inscripciones de la Facultad de Informática de la UNLP.
Respondé en español, breve y claro.
Usá siempre las herramientas para consultar datos reales. No inventes códigos, legajos, cupos, IDs ni nombres.
Para inscribir, verificá que el alumno y la materia existan y elegí un turno ofrecido: manana, tarde o noche.
La API valida cupo, correlativas, choque de turno, duplicados y materias aprobadas.
Si rechaza una inscripción, incluí en la respuesta final el código exacto del error
(por ejemplo MATERIA_APROBADA, SIN_CUPO, CORRELATIVA_PENDIENTE, CHOQUE_TURNO,
INSCRIPCION_DUPLICADA o TURNO_NO_DISPONIBLE) y el detalle. No reemplaces el código
por una frase equivalente.
Cuando tengas legajo, materia y turno y el usuario pida inscribirse, llamá a inscribir
y basá la respuesta en el resultado de la API. No rechaces por tu cuenta.
Para dar de baja, listá primero las inscripciones del alumno y usá el id exacto. Si hay varias, preguntá cuál.
No crees, edites ni borres alumnos o materias. No resetees la base.
Si falta el legajo, la materia o el turno, preguntá.
{legajo}"""

## 4. Herramientas (`agent.py:32-52`)

`TOOLS` es una lista de esquemas compatible con el tool calling de Ollama. Cada elemento describe una función que el modelo puede pedir.

En esta versión, `TOOLS` declara únicamente `obtener_alumno`:

| Herramienta | Declarada en `TOOLS` | Manejada en `ejecutar_herramienta` |
|---|---|---|
| `obtener_alumno` | Sí | Sí |
| `buscar_materias` | No | No |
| `obtener_materia` | No | No |
| `listar_inscripciones` | No | No |
| `inscribir` | No | No |
| `dar_de_baja` | No | No |

Cuando el modelo decide usar una herramienta, devuelve un `tool_call` con esta forma:

```json
{
    "function": {
        "name": "obtener_alumno",
        "arguments": {"legajo": "1001"}
    }
}
```

La `description` y el esquema de `parameters` son la información que el modelo usa para decidir cuándo llamar a la herramienta y cómo armar los argumentos. El esquema no se ejecuta: es documentación para el modelo.

In [6]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "obtener_alumno",
            "description": "Devuelve un alumno con sus materias aprobadas.",
            "parameters": {
                "type": "object",
                "properties": {
                    "legajo": {
                        "type": "string",
                        "description": "Legajo del alumno, por ejemplo 1001.",
                    }
                },
                "required": ["legajo"],
            },
        },
    },
    # TODO: agregar buscar_materias, obtener_materia, listar_inscripciones,
    # inscribir y dar_de_baja.
]

## 5. Helpers (`agent.py:55-77`)

- `_normalizar` pasa el texto a minúsculas y elimina los acentos. Se usa para comparar nombres de materias sin depender de mayúsculas ni tildes.
- `_debug_habilitado` considera habilitado el debug si `DEBUG` vale `1`, `true`, `yes` u `on`.
- `_debug` escribe en `stderr` con el prefijo `[debug]`.
- `_debug_api` agrega método, ruta, payload y status (o error de conexión) a la traza.

Ejemplo de salida con `DEBUG=1`:

```text
[debug] API GET /alumnos/1001 -> 200
[debug] API POST /inscripciones {"alumno": "1001", "materia": "CNE", "turno": "manana"} -> 409
```

In [7]:
def _normalizar(texto):
    return "".join(
        caracter
        for caracter in unicodedata.normalize("NFKD", str(texto).lower())
        if not unicodedata.combining(caracter)
    )


def _debug_habilitado():
    return os.environ.get("DEBUG", "").strip().lower() in {"1", "true", "yes", "on"}


def _debug(*partes):
    if _debug_habilitado():
        print("[debug]", *partes, file=sys.stderr, flush=True)


def _debug_api(method, path, payload, resultado):
    partes = [method, path]
    if payload is not None:
        partes.append(json.dumps(payload, ensure_ascii=False))
    partes.append(f"-> {resultado}")
    _debug("API", *partes)

## 6. Capa HTTP (`agent.py:80-98`)

`_api(method, path, payload=None)` es el único punto por el que el agente habla con la API. La llamada usa `requests.request(..., timeout=10)`.

| Situación | Resultado |
|---|---|
| Conexión fallida | `{"error": "No se pudo conectar a la API: <detalle>"}` |
| Status 204 | `{"ok": True}` |
| Status 2xx con JSON | El JSON parseado |
| Status 2xx sin JSON | El texto de la respuesta |
| Status 4xx o 5xx | `{"error": <cuerpo de la API>}` |

El error se devuelve como dato, no como excepción. Así el modelo recibe el rechazo de la API como resultado de la herramienta y puede explicarlo con el código exacto.

Ejemplo de error:

```json
{
    "error": {
        "code": "SIN_CUPO",
        "detail": "No hay cupo en CNE turno manana"
    }
}
```

Los parámetros que van en el path se codifican con `quote(..., safe="")` para evitar caracteres problemáticos en la URL.

In [8]:
def _api(method, path, payload=None):
    try:
        respuesta = requests.request(
            method, f"{API_URL}{path}", json=payload, timeout=10
        )
    except requests.RequestException as exc:
        _debug_api(method, path, payload, f"sin conexión: {exc}")
        return {"error": f"No se pudo conectar a la API: {exc}"}

    _debug_api(method, path, payload, respuesta.status_code)
    if respuesta.status_code == 204:
        return {"ok": True}
    try:
        cuerpo = respuesta.json()
    except ValueError:
        cuerpo = respuesta.text
    if respuesta.status_code >= 400:
        return {"error": cuerpo}
    return cuerpo

## 7. Dispatch (`agent.py:101-117`)

`ejecutar_herramienta(nombre, argumentos)` recibe lo que pidió el modelo y decide qué hacer:

1. Verifica que `argumentos` sea un diccionario.
2. Compara `nombre` contra las ramas implementadas.
3. Lee los argumentos con `argumentos.get(...)`.
4. Valida los obligatorios y devuelve un error claro si falta alguno.
5. Llama a `_api` con el método, la ruta y el payload correspondientes.
6. Si aparece `KeyError`, `TypeError` o `ValueError`, devuelve `{"error": "Argumentos inválidos: ..."}`.
7. Si ninguna rama coincide, devuelve `{"error": "Herramienta desconocida: <nombre>"}`.

La única rama implementada es `obtener_alumno`:

```python
if nombre == "obtener_alumno":
    legajo = argumentos.get("legajo")
    if not legajo:
        return {"error": "Falta el legajo"}
    return _api("GET", f"/alumnos/{quote(str(legajo), safe='')}")
```

Ejemplos:

| Llamada | Resultado |
|---|---|
| `ejecutar_herramienta("obtener_alumno", {"legajo": "1001"})` | JSON del alumno o `{"error": ...}` |
| `ejecutar_herramienta("obtener_alumno", {})` | `{"error": "Falta el legajo"}` |
| `ejecutar_herramienta("obtener_alumno", "1001")` | `{"error": "Argumentos inválidos"}` |
| `ejecutar_herramienta("no_existe", {})` | `{"error": "Herramienta desconocida: no_existe"}` |

In [9]:
def ejecutar_herramienta(nombre, argumentos):
    if not isinstance(argumentos, dict):
        return {"error": "Argumentos inválidos"}

    try:
        if nombre == "obtener_alumno":
            legajo = argumentos.get("legajo")
            if not legajo:
                return {"error": "Falta el legajo"}
            return _api("GET", f"/alumnos/{quote(str(legajo), safe='')}")

        # TODO: implementar buscar_materias, obtener_materia,
        # listar_inscripciones, inscribir y dar_de_baja.
    except (KeyError, TypeError, ValueError) as exc:
        return {"error": f"Argumentos inválidos: {exc}"}

    return {"error": f"Herramienta desconocida: {nombre}"}

## 8. Cliente de Ollama (`agent.py:120-150`)

`_ollama_chat(messages)` arma el payload y lo envía a `POST /api/chat` con `requests.post(..., timeout=120)`:

```json
{
    "model": "qwen3.5:9b",
    "messages": [...],
    "tools": [...],
    "stream": false,
    "think": false,
    "options": {"temperature": 0}
}
```

| Campo | Efecto |
|---|---|
| `model` | Modelo que responde |
| `messages` | Historial completo de la conversación |
| `tools` | Herramientas disponibles |
| `stream` | Una respuesta completa por pedido |
| `think` | Desactiva el modo de razonamiento de algunos modelos |
| `options.temperature` | Respuestas más deterministas |

La respuesta relevante tiene esta forma simplificada:

```json
{
    "message": {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "obtener_alumno",
                    "arguments": {"legajo": "1001"}
                }
            }
        ]
    }
}
```

Si no puede conectarse, el status no es 200, la respuesta no es JSON, trae un campo `error` o no trae `message`, la función devuelve `(None, "mensaje de error")`. Si todo sale bien, devuelve `(message, None)`.

In [10]:
def _ollama_chat(messages):
    payload = {
        "model": MODEL,
        "messages": messages,
        "tools": TOOLS,
        "stream": False,
        "think": False,
        "options": {"temperature": 0},
    }
    try:
        respuesta = requests.post(
            f"{OLLAMA_URL}/api/chat", json=payload, timeout=120
        )
    except requests.RequestException as exc:
        return None, f"No se pudo conectar a Ollama: {exc}"

    if _debug_habilitado():
        print("[debug]", respuesta.text, file=sys.stderr, flush=True)

    if respuesta.status_code != 200:
        return None, f"Ollama respondió {respuesta.status_code}: {respuesta.text}"
    try:
        datos = respuesta.json()
    except ValueError:
        return None, "Ollama devolvió una respuesta que no es JSON"
    if "error" in datos:
        return None, f"Ollama: {datos['error']}"
    mensaje = datos.get("message")
    if not isinstance(mensaje, dict):
        return None, "Ollama devolvió una respuesta sin mensaje"
    return mensaje, None

## 9. El bucle: `responder` (`agent.py:153-211`)

```mermaid
flowchart TD
    A["responder(messages)"] --> B{"¿Quedan pasos?"}
    B -->|Sí| C["_ollama_chat(messages)"]
    C -->|error| D["Devolver el error"]
    C -->|mensaje| E["Agregar mensaje assistant"]
    E --> F{"¿Hay tool_calls?"}
    F -->|No| G["Devolver content + códigos faltantes"]
    F -->|Sí| H["Recorrer cada tool_call"]
    H --> I["Parsear nombre y argumentos"]
    I --> J["ejecutar_herramienta"]
    J --> K["Agregar mensaje tool"]
    K --> L["Registrar error.code si existe"]
    L --> B
    B -->|No| M["Devolver fallback de MAX_PASOS"]
```

Paso a paso:

1. Inicializa `codigos_error` como una lista vacía.
2. Entra en un `for` de hasta `MAX_PASOS` iteraciones.
3. Llama a `_ollama_chat(messages)`. Si devuelve un error, lo retorna como texto y termina.
4. Arma un mensaje `assistant` con `content` y, si existen, `tool_calls`; lo agrega a `messages`.
5. Si no hay `tool_calls`, toma `content`, le aplica `.strip()` y lo devuelve. Si queda vacío, usa `"(sin respuesta)"`.
6. Si hay `tool_calls`, recorre cada una:
   - extrae `function.name` y `function.arguments`;
   - si `arguments` es un string, lo parsea con `json.loads`;
   - si el `tool_call` está mal formado, usa `{"error": "Tool call inválido"}`;
   - en caso contrario, llama a `ejecutar_herramienta(nombre, argumentos)`;
   - si el resultado es un diccionario con `error.code`, guarda ese código;
   - agrega un mensaje `role: "tool"` con el resultado serializado y `tool_name`.
7. Vuelve al paso 2 con el historial actualizado.
8. Si se agotan las iteraciones, devuelve el texto de fallback de `MAX_PASOS`.

### Roles de los mensajes

| `role` | `content` | Campos extra | Quién lo agrega |
|---|---|---|---|
| `system` | Reglas del asistente | — | `main` |
| `user` | Texto del usuario | — | `main` |
| `assistant` | Texto parcial o vacío | `tool_calls` si pidió herramientas | `responder` |
| `tool` | Resultado serializado en JSON | `tool_name` | `responder` |
| `assistant` | Respuesta final | — | `responder` |

El orden es parte del protocolo: el mensaje `assistant` con `tool_calls` va antes que el mensaje `tool` con el resultado. Si falta uno de los dos, el modelo pierde la relación entre el pedido y la respuesta.

### Respaldo de códigos de error

Mientras recorre las herramientas, el bucle guarda los códigos que devolvió la API. Cuando llega la respuesta final, compara cada código contra el texto normalizado. Si el modelo lo omitió, lo agrega al final:

```text
No se pudo inscribir.

Código de error: MATERIA_APROBADA
```

`codigos_error` es local a cada llamada de `responder`: no se comparte entre turnos.

In [ ]:
def responder(messages):
    codigos_error = []
    for _ in range(MAX_PASOS):
        mensaje, error = _ollama_chat(messages)
        if error:
            return error

        tool_calls = mensaje.get("tool_calls") or [] 
        mensaje_asistente = {
            "role": "assistant",
            "content": mensaje.get("content") or "",
        }
        if tool_calls:
            mensaje_asistente["tool_calls"] = tool_calls
        messages.append(mensaje_asistente)

        if not tool_calls:
            respuesta = mensaje_asistente["content"].strip() or "(sin respuesta)"
            faltantes = list(
                dict.fromkeys(
                    codigo
                    for codigo in codigos_error
                    if codigo.lower() not in _normalizar(respuesta)
                )
            )
            if faltantes:
                respuesta += "\n\nCódigo de error: " + ", ".join(faltantes)
            return respuesta

        for tool in tool_calls:
            nombre = "desconocida"
            argumentos = {}
            try:
                nombre = tool["function"]["name"]
                argumentos = tool["function"].get("arguments") or {}
                if isinstance(argumentos, str):
                    argumentos = json.loads(argumentos)
            except (KeyError, TypeError, AttributeError, json.JSONDecodeError):
                _debug("tool_call", json.dumps(tool, ensure_ascii=False))
                resultado = {"error": "Tool call inválido"}
            else:
                _debug("tool_call", nombre, json.dumps(argumentos, ensure_ascii=False))
                resultado = ejecutar_herramienta(nombre, argumentos)

            if isinstance(resultado, dict):
                error_tool = resultado.get("error")
                if isinstance(error_tool, dict) and error_tool.get("code"):
                    codigos_error.append(error_tool["code"])

            _debug("tool_result", nombre, json.dumps(resultado, ensure_ascii=False))
            messages.append(
                {
                    "role": "tool",
                    "content": json.dumps(resultado, ensure_ascii=False),
                    "tool_name": nombre,
                }
            )

    return "El agente superó el máximo de pasos sin dar una respuesta final."

## 10. Legajo actual y verificación de servicios (`agent.py:214-259`)

`_system_prompt()` completa el placeholder `{legajo}` del prompt. Los dos casos son:

```text
# LEGAJO_ACTUAL = "1001"
El alumno actual es 1001; usalo por defecto cuando no se indique otro.
```

```text
# LEGAJO_ACTUAL = None
No hay un alumno fijado; pedí el legajo cuando haga falta.
```

`_verificar_servicios()` se ejecuta al arrancar `main()` y comprueba:

| Verificación | Endpoint | Si falla |
|---|---|---|
| API disponible | `GET {API_URL}/openapi.json` | Muestra el error y devuelve `False` |
| Ollama disponible | `GET {OLLAMA_URL}/api/tags` | Muestra el error y devuelve `False` |
| Modelo descargado | Lista de `models` de Ollama | Muestra un aviso, pero continúa |

Cada verificación usa un timeout de 5 segundos.

In [12]:
def _system_prompt():
    if LEGAJO_ACTUAL:
        nota = (
            f"El alumno actual es {LEGAJO_ACTUAL}; "
            "usalo por defecto cuando no se indique otro."
        )
    else:
        nota = "No hay un alumno fijado; pedí el legajo cuando haga falta."
    return SYSTEM_PROMPT.format(legajo=nota)

In [13]:
def _verificar_servicios():
    try:
        respuesta = requests.get(f"{API_URL}/openapi.json", timeout=5)
    except requests.RequestException as exc:
        print(f"No se pudo conectar a la API en {API_URL}: {exc}", file=sys.stderr)
        return False
    if respuesta.status_code != 200:
        print(
            f"La API no responde en {API_URL} (HTTP {respuesta.status_code}).",
            file=sys.stderr,
        )
        return False

    try:
        respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    except requests.RequestException as exc:
        print(f"No se pudo conectar a Ollama en {OLLAMA_URL}: {exc}", file=sys.stderr)
        return False
    if respuesta.status_code != 200:
        print(
            f"Ollama no responde en {OLLAMA_URL} (HTTP {respuesta.status_code}).",
            file=sys.stderr,
        )
        return False

    try:
        modelos = {modelo.get("name") for modelo in respuesta.json().get("models", [])}
    except (ValueError, AttributeError):
        return True
    if MODEL not in modelos and f"{MODEL}:latest" not in modelos:
        print(
            f"Aviso: no veo {MODEL} en Ollama. Si falla, corré: ollama pull {MODEL}",
            file=sys.stderr,
        )
    return True

## 11. Entrada: `main` (`agent.py:262-293`)

`main()` arma el historial inicial:

```python
messages = [{"role": "system", "content": _system_prompt()}]
```

Después entra en un ciclo:

- si `stdin` es una terminal, usa `input("Vos: ")`;
- si la entrada viene por un pipe, lee con `sys.stdin.readline()`;
- ignora líneas vacías;
- termina con `/salir`, `salir`, `exit` o `quit`;
- por cada línea agrega un mensaje `user`, llama a `responder(messages)` e imprime el resultado.

El historial `messages` se reutiliza durante toda la sesión: `responder` lo modifica y lo deja con la respuesta final.

En la notebook **no ejecutamos `main()`** porque `input()` bloquea la celda. El bloque de entrada del archivo es:

```python
if __name__ == "__main__":
    sys.exit(main())
```

La demo de las próximas celdas llama directamente a `responder(messages)`, que es el mismo camino que sigue `main` después de leer una línea.

In [14]:
def main():
    if not _verificar_servicios():
        return 1

    messages = [{"role": "system", "content": _system_prompt()}]

    while True:
        if sys.stdin.isatty():
            try:
                linea = input("Vos: ")
            except (EOFError, KeyboardInterrupt):
                print()
                break
        else:
            linea = sys.stdin.readline()
            if not linea:
                break

        linea = linea.strip()
        if not linea:
            continue
        if linea.lower() in {"/salir", "salir", "exit", "quit"}:
            break

        messages.append({"role": "user", "content": linea})
        print(responder(messages), flush=True)

    return 0

## 12. Demo con servicios reales

Las próximas celdas usan la API y Ollama reales. La primera comprueba que los servicios estén disponibles; después muestra la capa HTTP, el dispatch y el bucle completo.

Qué mirar:

1. La diferencia entre llamar a `_api` directamente y llamar a `ejecutar_herramienta`.
2. El `tool_call` que produce Ollama y el mensaje `tool` que agrega `responder`.
3. Cómo el historial `messages` crece y conserva el contexto entre turnos.
4. La traza `[debug]` cuando `DEBUG=1`.

In [15]:
os.environ["DEBUG"] = "1"
listo = _verificar_servicios()
print("Servicios listos:" , listo)

Servicios listos: True


### Llamada directa a la API

`_api` no sabe nada del modelo: solo hace HTTP y devuelve el JSON. Esta celda muestra el alumno `1001` tal como lo entrega la API.

In [16]:
_api("GET", "/alumnos/1001")

[debug] API GET /alumnos/1001 -> 200


{'legajo': '1001',
 'nombre': 'Ana',
 'apellido': 'Pérez',
 'aprobadas': ['CNC', 'CNM', 'SI101', 'SI102', 'SI208', 'SI308', 'SI409']}

### Llamada directa a la herramienta

`ejecutar_herramienta` recibe el nombre que usaría el modelo y los argumentos. Para `obtener_alumno` termina llamando al mismo `_api` de la celda anterior.

También se puede ver el fallback para una herramienta que no está declarada en esta versión.

In [17]:
ejecutar_herramienta("obtener_alumno", {"legajo": "1001"})

[debug] API GET /alumnos/1001 -> 200


{'legajo': '1001',
 'nombre': 'Ana',
 'apellido': 'Pérez',
 'aprobadas': ['CNC', 'CNM', 'SI101', 'SI102', 'SI208', 'SI308', 'SI409']}

In [ ]:
ejecutar_herramienta(
    "inscribir",
    {"alumno": "1001", "materia": "CNE", "turno": "manana"},
)

### El bucle completo con Ollama

Esta celda arranca una conversación nueva, pregunta por el legajo `1001` y llama a `responder`. Con `DEBUG=1` se ven el `tool_call`, la llamada a la API y el resultado de la herramienta.

La respuesta final la produce el modelo; puede variar entre ejecuciones.

In [18]:
os.environ["DEBUG"] = "1"

messages = [
    {"role": "system", "content": _system_prompt()},
    {"role": "user", "content": "¿Qué datos tiene el legajo 1001?"},
]

print(responder(messages))

[debug] {"model":"qwen3.5:9b","created_at":"2026-11-21T11:28:06.177529849Z","message":{"role":"assistant","content":"","tool_calls":[{"id":"call_8y7w9tn7","function":{"index":0,"name":"obtener_alumno","arguments":{"legajo":"1001"}}}]},"done":true,"done_reason":"stop","total_duration":36828827736,"load_duration":8360580019,"prompt_eval_count":618,"prompt_eval_cached_count":0,"prompt_eval_duration":21682041000,"eval_count":33,"eval_duration":6774802000}
[debug] tool_call obtener_alumno {"legajo": "1001"}
[debug] API GET /alumnos/1001 -> 200
[debug] tool_result obtener_alumno {"legajo": "1001", "nombre": "Ana", "apellido": "Pérez", "aprobadas": ["CNC", "CNM", "SI101", "SI102", "SI208", "SI308", "SI409"]}
[debug] {"model":"qwen3.5:9b","created_at":"2026-11-21T11:28:25.043348017Z","message":{"role":"assistant","content":"El alumno con legajo 1001 es **Ana Pérez**.\n\nHa aprobado las siguientes materias:\n*   CNC\n*   CNM\n*   SI101\n*   SI102\n*   SI208\n*   SI308\n*   SI409"},"done":true,"

El alumno con legajo 1001 es **Ana Pérez**.

Ha aprobado las siguientes materias:
*   CNC
*   CNM
*   SI101
*   SI102
*   SI208
*   SI308
*   SI409


### Inspección del historial

Después de `responder`, `messages` contiene la conversación completa. Esta celda muestra el rol y un resumen de cada mensaje para ver la secuencia:

```text
system → user → assistant(tool_calls) → tool(resultado) → assistant(final)
```

In [ ]:
for i, mensaje in enumerate(messages):
    rol = mensaje["role"]
    contenido = (mensaje.get("content") or "").replace("\n", " ")
    extra = ""
    if mensaje.get("tool_calls"):
        nombres = [t["function"]["name"] for t in mensaje["tool_calls"]]
        extra = f" tool_calls={nombres}"
    if mensaje.get("tool_name"):
        extra = f" tool_name={mensaje['tool_name']}"
    print(f"{i}: {rol}{extra} -> {contenido[:160]!r}")

### Segundo turno

El mismo `messages` se reutiliza. Al agregar otro mensaje `user`, el modelo ve todo el historial anterior y puede responder usando el contexto de la primera pregunta.

In [ ]:
messages.append(
    {"role": "user", "content": "¿Cuáles son sus materias aprobadas?"}
)
print(responder(messages))

## 13. Cierre

El agente se puede leer como una cadena de responsabilidades:

1. `main` mantiene la conversación y la entrada/salida.
2. `_system_prompt` fija el rol y el legajo actual.
3. `TOOLS` declara lo que el modelo puede pedir.
4. `_ollama_chat` habla con el modelo.
5. `responder` coordina el bucle y conserva el historial.
6. `ejecutar_herramienta` traduce un `tool_call` a una operación.
7. `_api` ejecuta esa operación contra la API.

Preguntas para la clase:

- ¿Qué parte decide y qué parte ejecuta?
- ¿Por qué el resultado de la herramienta se agrega como mensaje `tool`?
- ¿Qué pasa si el modelo no incluye el código de error en la respuesta final?
- ¿Qué cambia si se agrega una herramienta nueva a `TOOLS`?
- ¿Por qué `MAX_PASOS` es necesario?